# Exploratory Data Analysis: NYC Short-Term Rental Prices

This notebook explores the raw `sample.csv` artifact and motivates the cleaning steps that live in `src/basic_cleaning/run.py`.

**Cleaning decisions arrived at here:**
1. Drop price outliers — keep rows where `min_price <= price <= max_price` (configured to $10–$350)
2. Convert `last_review` to a datetime dtype
3. Drop rows whose coordinates fall outside the NYC bounding box (a defensive guard for future samples; sample1 looks clean but `sample2.csv` contains stray points)

Each of these is justified below with summary stats and plots.

## 1. Setup

Import dependencies and start a W&B run with `job_type="eda"` so this analysis appears alongside the pipeline runs in W&B.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from ydata_profiling import ProfileReport

In [ ]:
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True, job_type="eda")

## 2. Fetch the sample artifact from W&B

Pull the most recent `sample.csv` produced by the `get_data` step. Using `run.use_artifact` (rather than reading a local file) records this exact artifact version as an input to this EDA run, which W&B will surface in the lineage graph.

In [ ]:
local_path = run.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.shape

## 3. First look

Shape, dtypes, and a few rows. The goal is to understand the schema before any quantitative analysis.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Observations from `describe()`:**
- `price` has an enormous range (`min` near 0, `max` in the thousands). This is suspicious for short-term rentals — likely data-entry errors or extreme luxury listings that aren't representative. We'll cap it.
- `latitude` and `longitude` look reasonable on their own (centered around NYC), but we'll check the extremes against the actual NYC bounding box in a moment.
- `reviews_per_month` and `last_review` have many missing values (visible in `info()` above) — properties that have never been reviewed. Sklearn's `SimpleImputer` handles this downstream.

## 4. Missing values

In [ ]:
df.isna().sum().sort_values(ascending=False)

`reviews_per_month` and `last_review` go together — they're both missing for listings that have never had a review. We won't drop these rows (they're a substantial fraction of the dataset); instead the modeling pipeline imputes them. `last_review` does need a dtype conversion though, which we do in `basic_cleaning`.

## 5. Price distribution → motivates the price filter

The `describe()` output suggested heavy-tailed price values. Let's visualize and pick reasonable cutoffs.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['price'].plot.hist(bins=100, ax=ax[0])
ax[0].set_title('Raw price distribution (linear scale)')
ax[0].set_xlabel('price ($/night)')
df['price'].plot.hist(bins=100, ax=ax[1], logy=True)
ax[1].set_title('Log-y scale (shows the long tail)')
ax[1].set_xlabel('price ($/night)')
plt.tight_layout()
plt.show()

In [ ]:
df['price'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

The bulk of the distribution sits between roughly $10 and $350 ($1st percentile to ~$95th). Beyond $350 the tail is sparse and likely either outliers or unrepresentative luxury listings.

**Decision:** filter to `10 <= price <= 350`. These bounds are stored in `config.yaml` (`etl.min_price`, `etl.max_price`) and applied by `basic_cleaning`.

## 6. Geographic coordinates → motivates the NYC bounding-box filter

Sample1 looks clean from `describe()` alone, but `data_check`'s `test_proper_boundaries` enforces the NYC bounds `longitude ∈ [-74.25, -73.50]` and `latitude ∈ [40.5, 41.2]`. Future samples (e.g. `sample2.csv`) are known to contain stray rows outside this box. To make `basic_cleaning` robust across future data we add a coordinate filter as well.

In [ ]:
in_bounds = df['longitude'].between(-74.25, -73.50) & df['latitude'].between(40.5, 41.2)
print(f"Rows in NYC bounding box: {in_bounds.sum()} / {len(df)}")
print(f"Rows outside (would be dropped): {(~in_bounds).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
df.plot.scatter(x='longitude', y='latitude', alpha=0.05, s=4, ax=ax)
ax.axvline(-74.25, color='r', linestyle='--', alpha=0.5)
ax.axvline(-73.50, color='r', linestyle='--', alpha=0.5)
ax.axhline(40.5, color='r', linestyle='--', alpha=0.5)
ax.axhline(41.2, color='r', linestyle='--', alpha=0.5)
ax.set_title('Listing locations vs. NYC bounding box (red)')
plt.show()

## 7. Datetime conversion

`last_review` arrives as a string. The training pipeline uses it via `delta_date_feature` which needs a proper datetime. We do the conversion in `basic_cleaning`.

In [ ]:
df['last_review'].head()

In [ ]:
pd.to_datetime(df['last_review']).head()

## 8. Categorical structure

Two categorical features matter for the downstream model: `neighbourhood_group` (5 NYC boroughs) and `room_type` (3 levels with a natural ordering).

In [ ]:
df['neighbourhood_group'].value_counts()

In [ ]:
df['room_type'].value_counts()

## 9. Full profiling report

Generate a ydata-profiling report for a single-page overview — useful as a reference artifact but not used for any specific decision below.

In [ ]:
profile = ProfileReport(df, title="NYC Airbnb raw sample", minimal=True)
profile.to_widgets()

## 10. Apply the cleaning rules and verify

Reproduce the three cleaning steps that `basic_cleaning` will run, and check the resulting dataframe.

In [ ]:
min_price, max_price = 10, 350

# 1. price filter
df_clean = df[df['price'].between(min_price, max_price)].copy()

# 2. datetime conversion
df_clean['last_review'] = pd.to_datetime(df_clean['last_review'])

# 3. NYC bounding-box filter
idx = df_clean['longitude'].between(-74.25, -73.50) & df_clean['latitude'].between(40.5, 41.2)
df_clean = df_clean[idx].copy()

print(f"Raw rows:     {len(df):>8,}")
print(f"Cleaned rows: {len(df_clean):>8,}")
print(f"Dropped:      {len(df) - len(df_clean):>8,}  ({(1 - len(df_clean)/len(df)):.1%})")

In [ ]:
df_clean.describe()

## 11. Close out

This EDA does **not** upload `clean_sample.csv` — that's the job of the `basic_cleaning` mlflow step. Closing the W&B run here just so this notebook shows up cleanly in the project's run list.

In [ ]:
run.finish()